##🧩	Exercício 1 – Impacto da Acentuação e da Remoção de Stopwords no spaCy
**Contexto:**
No notebook de exemplo, foi implementada a função `pipeline_spacy`, responsável por executar um pipeline de processamento textual utilizando a biblioteca spaCy.  
Esse pipeline inclui etapas de:  
- higienização do texto,  
-	normalização (com remoção opcional de acentuação),  
-	tokenização,  
-	remoção de stopwords,  
-	lematização.  

Observou-se que a remoção de acentuação pode causar perda de informação linguística, afetando diretamente o resultado da lematização. Por exemplo:
  
- Com acentuação removida:
  - “é” → “e” (perda da identidade verbal)  
- Com acentuação mantida:  
  - “é” → “ser” (lema correto)  

Esse comportamento evidencia que a normalização não é uma etapa neutra e que decisões inadequadas podem comprometer etapas posteriores do pipeline.  
Além disso, verificou-se que a lista padrão de stopwords do spaCy pode remover palavras semanticamente relevantes (como o adjetivo “bom”), reforçando a necessidade de customização da etapa de remoção de stopwords conforme o objetivo da aplicação.
Tarefa:  
Realize as seguintes modificações no código do pipeline:  
1.  Acentuação controlada: Modifique a função pipeline_spacy para que o usuário possa fornecer uma lista de palavras que devem manter a acentuação, mesmo quando a opção de remoção de acentos estiver ativada.  
2.  Remoção seletiva de stopwords: Altere a etapa de remoção de stopwords para:
o	remover apenas stopwords funcionais, como artigos, preposições e conjunções;
o	preservar palavras semanticamente relevantes, como adjetivos e verbos (por exemplo, "*bom*", "*ruim*", "*tem*").



In [ ]:
# Instalação do spaCy (Colab)
!pip -q install spacy


In [ ]:
# Download do modelo de português
!python -m spacy download pt_core_news_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 99.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import unicodedata
import spacy
# Carrega o modelo de linguagem para português
nlp = spacy.load("pt_core_news_sm")
print("spaCy carregado. Pipeline:", nlp.pipe_names)

spaCy carregado. Pipeline: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


1) `higienizar` foi mantida como estava

In [ ]:
def higienizar(texto: str) -> str:
    # remove HTML
    texto = re.sub(r"<[^>]+>", " ", texto)
    # remove URLs
    texto = re.sub(r"http\S+|www\.\S+", "", texto)
    # remove pontuação e símbolos (mantém letras/números/_ e espaços)
    texto = re.sub(r"[^\w\s]", " ", texto, flags=re.UNICODE)
    # normaliza espaços
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

2) `normalizar` foi alterada para acomodar as exceções

In [ ]:
import re
import unicodedata
from typing import Iterable, Dict

def _remover_acentos(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(c for c in s if not unicodedata.combining(c))

def normalizar(
    texto: str,
    remover_acentos: bool = True,
    manter_acentuadas: Iterable[str] = ()
) -> str:
    """
    Normaliza o texto:
    - lowercase
    - remove acentos do texto, exceto das palavras em 'manter_acentuadas'
    """
    texto = texto.lower()

    if not remover_acentos:
        return re.sub(r"\s+", " ", texto).strip()

    # cria placeholders para palavras que devem manter acento
    manter_acentuadas = [w.lower() for w in manter_acentuadas]
    placeholders: Dict[str, str] = {
        w: f"__KEEPACC_{i}__" for i, w in enumerate(manter_acentuadas)
    }

    # substitui palavras acentuadas por placeholders
    for palavra, ph in placeholders.items():
        texto = re.sub(rf"\b{re.escape(palavra)}\b", ph, texto)

    # remove acentos do restante do texto
    texto = _remover_acentos(texto)

    # restaura palavras preservadas
    for palavra, ph in placeholders.items():
        texto = texto.replace(ph, palavra)

    return re.sub(r"\s+", " ", texto).strip()


3) `remover_stopwords_space` ajustada para manter a acentuação

In [ ]:
def remover_stopwords_spacy(doc, manter_tokens=()):
    manter_tokens = set(manter_tokens)
    saida = []
    for t in doc:
        if t.text in manter_tokens:
            saida.append(t)
        elif t.is_stop and t.pos_ in {"DET", "ADP", "CCONJ", "SCONJ", "PRON"}:
            # remove apenas stopwords funcionais
            continue
        else:
            saida.append(t)
    return saida


4) `pipeline_spacy` ajustado para aceitar a lista

In [ ]:
def pipeline_spacy(
    texto_bruto: str,
    remover_acentos: bool = True,
    manter_acentuadas: list[str] | None = None
):
    manter_acentuadas = [w.lower() for w in (manter_acentuadas or [])]

    texto_limpo = higienizar(texto_bruto)
    texto_norm = normalizar(
        texto_limpo,
        remover_acentos=remover_acentos,
        manter_acentuadas=manter_acentuadas
    )

    doc = nlp(texto_norm)

    # stopwords (preserva tokens escolhidos)
    tokens_filtrados = remover_stopwords_spacy(doc, manter_tokens=manter_acentuadas)

    lemas = [t.lemma_ for t in tokens_filtrados]

    return {
        "texto_bruto": texto_bruto,
        "texto_normalizado": texto_norm,
        "tokens": [t.text for t in doc],
        "tokens_filtrados": [t.text for t in tokens_filtrados],
        "lemas": lemas
    }


5) `pipeline_spacy` alterada para acomodar as modificações

In [ ]:
texto = "O produto (http://exemplo.com) é bom ou ruim, mas não é <p title='Destaque'>perfeito</p> e tem defeitos"
# Remove acentos, mas preserva "é"
resultado = pipeline_spacy(
    texto,
    remover_acentos=True,
    manter_acentuadas=["é"]
)

print("Texto normalizado :", resultado["texto_normalizado"])
print("Tokens filtrados  :", resultado["tokens_filtrados"])
print("Lemas             :", resultado["lemas"])


Texto normalizado : o produto é bom ou ruim mas nao é perfeito e tem defeitos
Tokens filtrados  : ['produto', 'é', 'bom', 'ruim', 'nao', 'é', 'perfeito', 'tem', 'defeitos']
Lemas             : ['produto', 'ser', 'bom', 'ruim', 'nao', 'ser', 'perfeito', 'ter', 'defeito']
